In [1]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import ticker

# IEEE/Elsevier Double-Column Formatting Standard
plt.rcParams.update({
    'figure.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 7,
    'axes.labelsize': 7,
    'axes.titlesize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 6,
    'figure.titlesize': 8,
    'lines.linewidth': 1.0,
    'grid.linewidth': 0.5,
    'grid.alpha': 0.4
})

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from src.config import SimConfig, EnvConfig
from src.utils.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker, print_markdown_table
from src.solvers import AugmentedHybridSDPSolver
from src.plants import AugmentedHybridPlant
from src.controllers import build_approach, AugmentedValueControl

env = EnvConfig()
fleet_data = load_and_cache_entire_fleet(env)
exclude_days = [1, 2, 3]

value_factory = build_approach(
    controller_cls=AugmentedValueControl,
    plant_cls=AugmentedHybridPlant,
    solver_cls=AugmentedHybridSDPSolver,
    is_macro=False
)

def get_exact_complexity(delta_P, delta_t, N_d, delta_n):
    cfg = SimConfig(delta_P=delta_P, delta_t=delta_t, N_d=N_d, use_smart_grid=True, delta_n=delta_n, apply_terminal_soc_cost=True, alpha_fc=4, verbose=False)
    N_fc = len(cfg.n_vals)
    N_s = cfg.N_s
    N_f = len(cfg.pfc_vals)
    N_b = len(cfg.pb_vals)
    S_nodes = N_d * N_fc * N_s * N_f
    A_nodes = N_fc * N_b
    steps = 86400 / delta_t
    return steps * S_nodes * (N_d + A_nodes)

# We use the lighter CPU budget so you can churn through points quickly today
OLD_TARGET = get_exact_complexity(delta_P=80.0, delta_t=180, N_d=5, delta_n=3)
GLOBAL_TARGET_COMPLEXITY = OLD_TARGET / 10.0
print(f"CPU BUDGET LOCKED AT: {GLOBAL_TARGET_COMPLEXITY:.2e} operations/day")

Loading fleet data from binary cache (fleet_data_cache_scale_0.75.npz)...
 -> Cache instantly loaded in 0.456 seconds.
CPU BUDGET LOCKED AT: 3.77e+08 operations/day


In [2]:
import itertools

seconds_in_day = 86400
dt_options = sorted([d for d in range(1, 3601) if seconds_in_day % d == 0])

# Restricting delta_P to clean multiples of 5 to avoid extreme decimal weirdness, 
# but still allowing inharmonic values to capture the full grid resonance effect.
dp_options = list(range(5, 1001, 5)) 
npd_options = list(range(2, 35))
npack_options = [1, 2, 3, 4]
MAX_SHIP_POWER = 1800.0 

print("Hunting for valid iso-complexity configurations...")

valid_configs = []
for dt, dp, npd, npack in itertools.product(dt_options, dp_options, npd_options, npack_options):
    if npd > int(MAX_SHIP_POWER / dp) + 1:
        continue 
        
    comp = get_exact_complexity(dp, dt, npd, npack)
    variance = abs(comp - GLOBAL_TARGET_COMPLEXITY) / GLOBAL_TARGET_COMPLEXITY
    
    # Keeping variance tight for fair comparison
    if variance <= 0.05:
        ratio = 24000 / (npack * dp * npd)
        valid_configs.append({
            'delta_t': dt, 'delta_P': dp, 'N_d': npd, 'delta_n': npack, 
            'Ratio': ratio, 'Variance (%)': variance * 100
        })

df_all_valid = pd.DataFrame(valid_configs)

# --- THE REPRODUCIBLE SHUFFLE ---
# By using random_state=42, this list will ALWAYS be in the exact same order.
df_shuffled = df_all_valid.sample(frac=1.0, random_state=42).reset_index(drop=True)

# ---> SET HOW MANY POINTS YOU WANT TO RUN HERE <---
TARGET_POINTS = 400 
df_target = df_shuffled.head(TARGET_POINTS)

print(f"Found {len(df_all_valid)} total valid configurations.")
print(f"Targeting the first {TARGET_POINTS} points of the reproducible sequence.")

Hunting for valid iso-complexity configurations...
Found 2735 total valid configurations.
Targeting the first 400 points of the reproducible sequence.


In [4]:
csv_filename = "cost_vs_dt_scatter.csv"

# Load existing results to save time if we are expanding the run
if os.path.exists(csv_filename):
    df_existing = pd.read_csv(csv_filename)
    print(f"Loaded {len(df_existing)} previously computed results from {csv_filename}.")
else:
    df_existing = pd.DataFrame()
    print("No existing save file found. Starting fresh.")

all_results = df_existing.to_dict('records')

# Helper function to check if a config was already computed
def is_computed(dt, dp, npd, npack, df_hist):
    if df_hist.empty: return False
    match = df_hist[(df_hist['delta_t'] == dt) & (df_hist['delta_P'] == dp) & 
                    (df_hist['N_d'] == npd) & (df_hist['delta_n'] == npack)]
    return not match.empty

runs_to_do = []
for _, row in df_target.iterrows():
    if not is_computed(row['delta_t'], row['delta_P'], row['N_d'], row['delta_n'], df_existing):
        runs_to_do.append(row)

print(f"--- COMMENCING SCATTER RUN ({len(runs_to_do)} new configurations to compute) ---")

for i, row in enumerate(runs_to_do):
    dt, dp, npd, npack = int(row['delta_t']), int(row['delta_P']), int(row['N_d']), int(row['delta_n'])
    print(f"Run [{i+1}/{len(runs_to_do)}] -> delta_t={dt}s | delta_P={dp} | N_d={npd} | delta_n={npack}")
    
    cfg = SimConfig(delta_P=dp, delta_t=dt, N_d=npd, use_smart_grid=True, delta_n=npack, apply_terminal_soc_cost=True, alpha_fc=4, verbose=False)
    bm = VoyageBenchmarker(fleet_data, env, cfg, exclude_days)
    report = bm.run_leave_one_out(value_factory)
    
    res = row.to_dict()
    res['Total Cost [$]'] = report.summary.loc['Average', 'Total Cost [$]']
    res['Offline Compute [s]'] = report.summary.loc['Average', 'Offline Compute Time [s]']
    all_results.append(res)
    
    # Iterative save
    pd.DataFrame(all_results).to_csv(csv_filename, index=False)

df_final_results = pd.DataFrame(all_results)
print(f"\nSaved {len(df_final_results)} total points to {csv_filename}.")

Loaded 418 previously computed results from cost_vs_dt_scatter.csv.
--- COMMENCING SCATTER RUN (227 new configurations to compute) ---
Run [1/227] -> delta_t=16s | delta_P=875 | N_d=3 | delta_n=4
 [Vault] Cache HIT: Markov Chain loaded (geometric) for days [5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


KeyboardInterrupt: 

In [ ]:
from matplotlib import ticker

# ---> COST NORMALIZATION TOGGLE <---
NORMALIZE_COST = False  # Set to False to plot raw Dollars ($)

df_plot = pd.read_csv("cost_vs_dt_scatter.csv")

os.makedirs('figures', exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 4))

# Dynamically map a color gradient to delta_n just to give the cloud some extra visual dimension
npack_vals = sorted(df_plot['delta_n'].unique())
cmap = plt.get_cmap('viridis')
colors = {n: cmap(i / max(1, len(npack_vals)-1)) for i, n in enumerate(npack_vals)}

if NORMALIZE_COST:
    min_cost = df_plot['Total Cost [$]'].min()
    df_plot['Y_Value'] = ((df_plot['Total Cost [$]'] - min_cost) / min_cost) * 100
    y_label = 'Normalized Cost Penalty (%)'
    title = 'Cost Penalty vs. ZOH Update Rate (Iso-Complexity)'
else:
    df_plot['Y_Value'] = df_plot['Total Cost [$]']
    y_label = 'Total Operational Cost [$]'
    title = 'Total Cost vs. ZOH Update Rate (Iso-Complexity)'

for n in npack_vals:
    subset = df_plot[df_plot['delta_n'] == n]
    if not subset.empty:
        ax.scatter(subset['delta_t'], subset['Y_Value'], 
                   color=colors[n], alpha=0.7, edgecolors='k', linewidth=0.5,
                   label=f'$n_{{pack}} = {int(n)}$')

# Optional: Draw a Pareto Front boundary on the bottom of the cloud
df_sorted = df_plot.sort_values('delta_t')
rolling_min = df_sorted.groupby('delta_t')['Y_Value'].min()
ax.plot(rolling_min.index, rolling_min.values, color='black', linestyle='--', linewidth=1.5, alpha=0.6, label='Empirical Pareto Front')

ax.set_xlabel('Temporal Resolution ($\\Delta t$ [s])')
ax.set_ylabel(y_label)
ax.set_title(title, pad=10)
ax.set_xscale('log')
ax.set_xticks([60, 90, 120, 180, 240, 300, 600, 1200, 1800, 3600])
ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())

if NORMALIZE_COST:
    ax.set_ylim(bottom=-0.5) 

ax.grid(True, which='both', ls=':', alpha=0.5)
ax.legend(title="Module Grouping", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
filename_save = 'figures/06_scatter_dt_norm.png' if NORMALIZE_COST else 'figures/06_scatter_dt_abs.png'
plt.savefig(filename_save, dpi=300, bbox_inches='tight')
plt.show()